# Actividad 6: Implementación de AFD y AFND

**Equipo**

| Nombre completo | No. de cuenta |
|---|---|
| Jonathan Hernández Lazcano | 200417 |
| Camila Rodriguez Rosas | 194100 |

Este notebook resuelve los ejercicios llamando a los **endpoints reales** de la API FastAPI
definida en `main.py`. Se usa `TestClient` para invocarlos sin necesidad de levantar un servidor
aparte, de modo que lo que se documenta aquí es exactamente lo que devuelve la API.

En las peticiones la cadena vacía $\lambda$ se escribe como `""`, y en los resultados se muestra
como `λ`. En las tablas de transición, `→` marca el estado inicial y `*` los estados de aceptación.

## Preparación

Se crea el cliente y dos ayudantes: `evaluar` hace el POST al endpoint que corresponda y
`resumen` formatea la respuesta como una tabla de Markdown.

In [1]:
import sys
import warnings

sys.path.append("..")
warnings.filterwarnings("ignore")

from fastapi.testclient import TestClient
from IPython.display import Markdown

from main import app

client = TestClient(app)


def evaluar(tipo, automata):
    """POST a /afd/evaluar o /afnd/evaluar; devuelve el JSON de la respuesta."""
    respuesta = client.post(f"/{tipo}/evaluar", json=automata)
    respuesta.raise_for_status()
    return respuesta.json()


def conjunto(elementos):
    return "{" + ", ".join(elementos) + "}" if elementos else "∅"


def resumen(datos):
    """Presenta la respuesta de la API como Markdown."""
    lineas = [
        f"**Tipo de autómata:** {datos['tipo']}  ",
        f"**Estados totales (Q):** {conjunto(datos['estados'])}  ",
        f"**Alfabeto (Σ):** {conjunto(datos['alfabeto'])}  ",
        f"**Estado inicial (q₀):** {datos['estado_inicial']}  ",
        f"**Estados finales (F):** {conjunto(datos['estados_finales'])}",
        "",
    ]

    con_camino = any(r.get("camino_aceptacion") for r in datos["resultados"])
    encabezado = "| Cadena | ¿Aceptada? | Notación de transición |"
    separador = "|---|:--:|---|"
    if con_camino:
        encabezado += " Camino de aceptación |"
        separador += "---|"
    lineas += [encabezado, separador]

    for r in datos["resultados"]:
        veredicto = "✅ Sí" if r["aceptada"] else "❌ No"
        fila = f"| `{r['cadena_mostrada']}` | {veredicto} | `{r['notacion']}` |"
        if con_camino:
            camino = r.get("camino_aceptacion")
            fila += f" `{camino}` |" if camino else " — |"
        lineas.append(fila)

    return Markdown("\n".join(lineas))

## Ejercicio 1 (AFD) — Cadenas con un número par de `a`

$\Sigma = \{a, b\}$ &nbsp;&nbsp; $L = \{\, w \in \Sigma^* : w \text{ tiene una cantidad par de } a \,\}$

La idea es guardar en el estado la paridad de las `a` leídas hasta el momento: `q0` significa
«llevo un número par» (y es de aceptación) y `q1` «llevo un número impar». Cada `b` no cambia
la paridad, así que es un bucle.

**Tabla de transición**

| δ | a | b |
|---|---|---|
| → *q0* | q1 | q0 |
| q1 | q0 | q1 |

In [2]:
afd_paridad = {
    "tabla": {
        "q0": {"a": "q1", "b": "q0"},
        "q1": {"a": "q0", "b": "q1"},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q0"],
    "cadenas": ["", "a", "aa", "ba", "abab", "aaa", "bbb"],
}

resultado_1 = evaluar("afd", afd_paridad)

In [3]:
resumen(resultado_1)

**Tipo de autómata:** AFD  
**Estados totales (Q):** {q0, q1}  
**Alfabeto (Σ):** {a, b}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q0}

| Cadena | ¿Aceptada? | Notación de transición |
|---|:--:|---|
| `λ` | ✅ Sí | `(q0, λ)` |
| `a` | ❌ No | `(q0, a) ⊢ (q1, λ)` |
| `aa` | ✅ Sí | `(q0, aa) ⊢ (q1, a) ⊢ (q0, λ)` |
| `ba` | ❌ No | `(q0, ba) ⊢ (q0, a) ⊢ (q1, λ)` |
| `abab` | ✅ Sí | `(q0, abab) ⊢ (q1, bab) ⊢ (q1, ab) ⊢ (q0, b) ⊢ (q0, λ)` |
| `aaa` | ❌ No | `(q0, aaa) ⊢ (q1, aa) ⊢ (q0, a) ⊢ (q1, λ)` |
| `bbb` | ✅ Sí | `(q0, bbb) ⊢ (q0, bb) ⊢ (q0, b) ⊢ (q0, λ)` |

## Ejercicio 2 (AFD) — Números binarios múltiplos de 3

$\Sigma = \{0, 1\}$ &nbsp;&nbsp; $L = \{\, w \in \Sigma^* : w \text{ leído en binario es múltiplo de } 3 \,\}$

Cada estado representa el residuo del número leído hasta ahora módulo 3. Al leer un bit $b$ el
número se duplica y se le suma el bit, así que la transición es
$\delta(r_i, b) = r_{(2i + b) \bmod 3}$. Se acepta cuando el residuo es 0.

**Tabla de transición**

| δ | 0 | 1 |
|---|---|---|
| → *r0* | r0 | r1 |
| r1 | r2 | r0 |
| r2 | r1 | r2 |

In [4]:
afd_multiplos_de_3 = {
    "tabla": {
        "r0": {"0": "r0", "1": "r1"},
        "r1": {"0": "r2", "1": "r0"},
        "r2": {"0": "r1", "1": "r2"},
    },
    "estado_inicial": "r0",
    "estados_finales": ["r0"],
    "cadenas": ["0", "10", "11", "110", "101", "1001", "1111"],
}

resultado_2 = evaluar("afd", afd_multiplos_de_3)

In [5]:
resumen(resultado_2)

**Tipo de autómata:** AFD  
**Estados totales (Q):** {r0, r1, r2}  
**Alfabeto (Σ):** {0, 1}  
**Estado inicial (q₀):** r0  
**Estados finales (F):** {r0}

| Cadena | ¿Aceptada? | Notación de transición |
|---|:--:|---|
| `0` | ✅ Sí | `(r0, 0) ⊢ (r0, λ)` |
| `10` | ❌ No | `(r0, 10) ⊢ (r1, 0) ⊢ (r2, λ)` |
| `11` | ✅ Sí | `(r0, 11) ⊢ (r1, 1) ⊢ (r0, λ)` |
| `110` | ✅ Sí | `(r0, 110) ⊢ (r1, 10) ⊢ (r0, 0) ⊢ (r0, λ)` |
| `101` | ❌ No | `(r0, 101) ⊢ (r1, 01) ⊢ (r2, 1) ⊢ (r2, λ)` |
| `1001` | ✅ Sí | `(r0, 1001) ⊢ (r1, 001) ⊢ (r2, 01) ⊢ (r1, 1) ⊢ (r0, λ)` |
| `1111` | ✅ Sí | `(r0, 1111) ⊢ (r1, 111) ⊢ (r0, 11) ⊢ (r1, 1) ⊢ (r0, λ)` |

## Ejercicio 3 (AFD) — Cadenas que terminan en `ab`

$\Sigma = \{a, b\}$ &nbsp;&nbsp; $L = \{\, wab : w \in \Sigma^* \,\}$

Aquí el estado recuerda cuánto del sufijo `ab` se lleva reconocido: `q0` nada, `q1` que la
última letra fue una `a`, y `q2` que la cadena termina justo en `ab`. Nótese que desde `q2`
una `a` regresa a `q1` (no a `q0`), porque esa `a` puede iniciar un nuevo sufijo.

**Tabla de transición**

| δ | a | b |
|---|---|---|
| → q0 | q1 | q0 |
| q1 | q1 | q2 |
| * q2 | q1 | q0 |

In [6]:
afd_termina_en_ab = {
    "tabla": {
        "q0": {"a": "q1", "b": "q0"},
        "q1": {"a": "q1", "b": "q2"},
        "q2": {"a": "q1", "b": "q0"},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q2"],
    "cadenas": ["ab", "aab", "abb", "bab", "a", "abab", "ba"],
}

resultado_3 = evaluar("afd", afd_termina_en_ab)

In [7]:
resumen(resultado_3)

**Tipo de autómata:** AFD  
**Estados totales (Q):** {q0, q1, q2}  
**Alfabeto (Σ):** {a, b}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q2}

| Cadena | ¿Aceptada? | Notación de transición |
|---|:--:|---|
| `ab` | ✅ Sí | `(q0, ab) ⊢ (q1, b) ⊢ (q2, λ)` |
| `aab` | ✅ Sí | `(q0, aab) ⊢ (q1, ab) ⊢ (q1, b) ⊢ (q2, λ)` |
| `abb` | ❌ No | `(q0, abb) ⊢ (q1, bb) ⊢ (q2, b) ⊢ (q0, λ)` |
| `bab` | ✅ Sí | `(q0, bab) ⊢ (q0, ab) ⊢ (q1, b) ⊢ (q2, λ)` |
| `a` | ❌ No | `(q0, a) ⊢ (q1, λ)` |
| `abab` | ✅ Sí | `(q0, abab) ⊢ (q1, bab) ⊢ (q2, ab) ⊢ (q1, b) ⊢ (q2, λ)` |
| `ba` | ❌ No | `(q0, ba) ⊢ (q0, a) ⊢ (q1, λ)` |

## Ejercicio 4 (AFND) — El lenguaje $(a|b)^*abb$

$\Sigma = \{a, b\}$ &nbsp;&nbsp; $L = L((a|b)^*abb)$

El no determinismo está concentrado en $\delta(q_0, a) = \{q_0, q_1\}$: al leer una `a` el
autómata puede quedarse en `q0` (esa `a` es parte del prefijo cualquiera) o «apostar» a que
justo ahí empieza el sufijo `abb` y pasar a `q1`. Como la evaluación explora todas las ramas
a la vez, basta con que **una** llegue a `q3`.

**Tabla de transición**

| δ | a | b |
|---|---|---|
| → q0 | {q0, q1} | {q0} |
| q1 | ∅ | {q2} |
| q2 | ∅ | {q3} |
| * q3 | ∅ | ∅ |

In [8]:
afnd_abb = {
    "tabla": {
        "q0": {"a": ["q0", "q1"], "b": ["q0"]},
        "q1": {"b": ["q2"]},
        "q2": {"b": ["q3"]},
        "q3": {},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q3"],
    "cadenas": ["abb", "aabb", "babb", "ab", "abba", "bb"],
}

resultado_4 = evaluar("afnd", afnd_abb)

In [9]:
resumen(resultado_4)

**Tipo de autómata:** AFND  
**Estados totales (Q):** {q0, q1, q2, q3}  
**Alfabeto (Σ):** {a, b}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q3}

| Cadena | ¿Aceptada? | Notación de transición | Camino de aceptación |
|---|:--:|---|---|
| `abb` | ✅ Sí | `{q0} --a--> {q0, q1} --b--> {q0, q2} --b--> {q0, q3}` | `q0 --a--> q1 --b--> q2 --b--> q3` |
| `aabb` | ✅ Sí | `{q0} --a--> {q0, q1} --a--> {q0, q1} --b--> {q0, q2} --b--> {q0, q3}` | `q0 --a--> q0 --a--> q1 --b--> q2 --b--> q3` |
| `babb` | ✅ Sí | `{q0} --b--> {q0} --a--> {q0, q1} --b--> {q0, q2} --b--> {q0, q3}` | `q0 --b--> q0 --a--> q1 --b--> q2 --b--> q3` |
| `ab` | ❌ No | `{q0} --a--> {q0, q1} --b--> {q0, q2}` | — |
| `abba` | ❌ No | `{q0} --a--> {q0, q1} --b--> {q0, q2} --b--> {q0, q3} --a--> {q0, q1}` | — |
| `bb` | ❌ No | `{q0} --b--> {q0} --b--> {q0}` | — |

## Ejercicio 5 (AFND) — Cadenas que terminan en `aa` o en `bb`

$\Sigma = \{a, b\}$ &nbsp;&nbsp; $L = \{\, w\,aa : w \in \Sigma^* \,\} \cup \{\, w\,bb : w \in \Sigma^* \,\}$

Desde `q0` salen dos ramas independientes: `q1 → q2` verifica el sufijo `aa` y `q3 → q4`
verifica el sufijo `bb`. Este es el patrón típico para expresar una **unión** de lenguajes con
un AFND: se ponen las dos máquinas en paralelo y el estado inicial adivina cuál usar.

**Tabla de transición**

| δ | a | b |
|---|---|---|
| → q0 | {q0, q1} | {q0, q3} |
| q1 | {q2} | ∅ |
| * q2 | ∅ | ∅ |
| q3 | ∅ | {q4} |
| * q4 | ∅ | ∅ |

In [10]:
afnd_aa_o_bb = {
    "tabla": {
        "q0": {"a": ["q0", "q1"], "b": ["q0", "q3"]},
        "q1": {"a": ["q2"]},
        "q2": {},
        "q3": {"b": ["q4"]},
        "q4": {},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q2", "q4"],
    "cadenas": ["aa", "bb", "abaa", "abb", "ab", "b", "baab"],
}

resultado_5 = evaluar("afnd", afnd_aa_o_bb)

In [11]:
resumen(resultado_5)

**Tipo de autómata:** AFND  
**Estados totales (Q):** {q0, q1, q2, q3, q4}  
**Alfabeto (Σ):** {a, b}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q2, q4}

| Cadena | ¿Aceptada? | Notación de transición | Camino de aceptación |
|---|:--:|---|---|
| `aa` | ✅ Sí | `{q0} --a--> {q0, q1} --a--> {q0, q1, q2}` | `q0 --a--> q1 --a--> q2` |
| `bb` | ✅ Sí | `{q0} --b--> {q0, q3} --b--> {q0, q3, q4}` | `q0 --b--> q3 --b--> q4` |
| `abaa` | ✅ Sí | `{q0} --a--> {q0, q1} --b--> {q0, q3} --a--> {q0, q1} --a--> {q0, q1, q2}` | `q0 --a--> q0 --b--> q0 --a--> q1 --a--> q2` |
| `abb` | ✅ Sí | `{q0} --a--> {q0, q1} --b--> {q0, q3} --b--> {q0, q3, q4}` | `q0 --a--> q0 --b--> q3 --b--> q4` |
| `ab` | ❌ No | `{q0} --a--> {q0, q1} --b--> {q0, q3}` | — |
| `b` | ❌ No | `{q0} --b--> {q0, q3}` | — |
| `baab` | ❌ No | `{q0} --b--> {q0, q3} --a--> {q0, q1} --a--> {q0, q1, q2} --b--> {q0, q3}` | — |

## Ejercicio 6 (AFND-λ) — El lenguaje $a^*b^*c^*$

$\Sigma = \{a, b, c\}$ &nbsp;&nbsp; $L = L(a^*b^*c^*)$

Cada estado es el bucle de un bloque (`q0` para las `a`, `q1` para las `b`, `q2` para las `c`) y
las **transiciones λ** conectan un bloque con el siguiente **sin consumir símbolos**. Gracias a
ellas cualquier bloque puede quedar vacío: la clausura-λ de $\{q_0\}$ es $\{q_0, q_1, q_2\}$, y
como `q2` es de aceptación, la cadena vacía se acepta de inmediato.

**Tabla de transición**

| δ | a | b | c | λ |
|---|---|---|---|---|
| → q0 | {q0} | ∅ | ∅ | {q1} |
| q1 | ∅ | {q1} | ∅ | {q2} |
| * q2 | ∅ | ∅ | {q2} | ∅ |

> La columna `λ` es opcional y **no** forma parte del alfabeto Σ. La API también acepta
> escribirla como `"lambda"`, `"ε"` o `""`.

In [12]:
afnd_lambda = {
    "tabla": {
        "q0": {"a": ["q0"], "λ": ["q1"]},
        "q1": {"b": ["q1"], "λ": ["q2"]},
        "q2": {"c": ["q2"]},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q2"],
    "cadenas": ["", "aaa", "aabbcc", "ac", "bc", "abc", "ba", "cba"],
}

resultado_6 = evaluar("afnd", afnd_lambda)

In [13]:
resumen(resultado_6)

**Tipo de autómata:** AFND-λ  
**Estados totales (Q):** {q0, q1, q2}  
**Alfabeto (Σ):** {a, b, c}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q2}

| Cadena | ¿Aceptada? | Notación de transición | Camino de aceptación |
|---|:--:|---|---|
| `λ` | ✅ Sí | `{q0, q1, q2}` | `q0 --λ--> q1 --λ--> q2` |
| `aaa` | ✅ Sí | `{q0, q1, q2} --a--> {q0, q1, q2} --a--> {q0, q1, q2} --a--> {q0, q1, q2}` | `q0 --a--> q0 --a--> q0 --a--> q0 --λ--> q1 --λ--> q2` |
| `aabbcc` | ✅ Sí | `{q0, q1, q2} --a--> {q0, q1, q2} --a--> {q0, q1, q2} --b--> {q1, q2} --b--> {q1, q2} --c--> {q2} --c--> {q2}` | `q0 --a--> q0 --a--> q0 --λ--> q1 --b--> q1 --b--> q1 --λ--> q2 --c--> q2 --c--> q2` |
| `ac` | ✅ Sí | `{q0, q1, q2} --a--> {q0, q1, q2} --c--> {q2}` | `q0 --a--> q0 --λ--> q1 --λ--> q2 --c--> q2` |
| `bc` | ✅ Sí | `{q0, q1, q2} --b--> {q1, q2} --c--> {q2}` | `q0 --λ--> q1 --b--> q1 --λ--> q2 --c--> q2` |
| `abc` | ✅ Sí | `{q0, q1, q2} --a--> {q0, q1, q2} --b--> {q1, q2} --c--> {q2}` | `q0 --a--> q0 --λ--> q1 --b--> q1 --λ--> q2 --c--> q2` |
| `ba` | ❌ No | `{q0, q1, q2} --b--> {q1, q2} --a--> ∅` | — |
| `cba` | ❌ No | `{q0, q1, q2} --c--> {q2} --b--> ∅` | — |

## Validaciones y casos especiales

Los endpoints no solo evalúan: también verifican que la tabla describa un autómata bien formado.
Hay dos niveles de error:

1. **Error de definición del autómata** → `HTTP 422`, no se evalúa nada.
2. **Problema con una cadena concreta** (símbolo fuera de Σ, transición indefinida) → esa cadena
   se rechaza indicando el motivo, pero las demás del lote sí se evalúan.

In [14]:
casos_invalidos = [
    ("Estado inicial inexistente", "afd", {
        "tabla": {"q0": {"a": "q0"}},
        "estado_inicial": "qX", "estados_finales": ["q0"], "cadenas": ["a"]}),
    ("Estado final inexistente", "afd", {
        "tabla": {"q0": {"a": "q0"}},
        "estado_inicial": "q0", "estados_finales": ["qZ"], "cadenas": ["a"]}),
    ("AFD con dos destinos en una celda", "afd", {
        "tabla": {"q0": {"a": ["q0", "q1"]}, "q1": {}},
        "estado_inicial": "q0", "estados_finales": ["q1"], "cadenas": ["a"]}),
    ("AFD con transición λ", "afd", {
        "tabla": {"q0": {"a": "q0", "λ": "q1"}, "q1": {}},
        "estado_inicial": "q0", "estados_finales": ["q1"], "cadenas": ["a"]}),
]

for titulo, tipo, automata in casos_invalidos:
    respuesta = client.post(f"/{tipo}/evaluar", json=automata)
    print(f"{titulo}: HTTP {respuesta.status_code}")
    print(f"    {respuesta.json()['detail']}")

Estado inicial inexistente: HTTP 422
    El estado inicial 'qX' no aparece en la tabla de transición.
Estado final inexistente: HTTP 422
    Estado(s) final(es) que no aparecen en la tabla de transición: qZ
AFD con dos destinos en una celda: HTTP 422
    Un AFD debe tener a lo más un destino por celda; hay varios en: δ(q0, a)
AFD con transición λ: HTTP 422
    Un AFD no admite transiciones λ; usa el endpoint /afnd/evaluar.


### AFD incompleto y símbolos fuera del alfabeto

En el AFD siguiente no existe $\delta(q_0, b)$, así que `ba` muere en la primera transición; y
`abc` contiene una `c`, que no pertenece a $\Sigma = \{a, b\}$. En ambos casos la notación indica
con ∅ el punto exacto donde se detuvo el recorrido, mientras que `aab` se evalúa normalmente.

In [15]:
afd_incompleto = {
    "tabla": {
        "q0": {"a": "q1"},
        "q1": {"a": "q1", "b": "q1"},
    },
    "estado_inicial": "q0",
    "estados_finales": ["q1"],
    "cadenas": ["ba", "aab", "abc"],
}

resumen(evaluar("afd", afd_incompleto))

**Tipo de autómata:** AFD  
**Estados totales (Q):** {q0, q1}  
**Alfabeto (Σ):** {a, b}  
**Estado inicial (q₀):** q0  
**Estados finales (F):** {q1}

| Cadena | ¿Aceptada? | Notación de transición |
|---|:--:|---|
| `ba` | ❌ No | `(q0, ba) ⊢ (∅, a)` |
| `aab` | ✅ Sí | `(q0, aab) ⊢ (q1, ab) ⊢ (q1, b) ⊢ (q1, λ)` |
| `abc` | ❌ No | `(q0, abc) ⊢ (q1, bc) ⊢ (q1, c) ⊢ (∅, λ)` |

In [16]:
# El motivo detallado de cada cadena también viaja en la respuesta
for r in evaluar("afd", afd_incompleto)["resultados"]:
    print(f"{r['cadena_mostrada']:<5} -> {r['motivo']}")

ba    -> No hay transición definida para δ(q0, b).
aab   -> Termina en q1, que es un estado de aceptación.
abc   -> El símbolo 'c' no pertenece al alfabeto Σ = {a, b}.


## Conclusiones

- El **AFD** se evalúa con un recorrido lineal sobre la cadena: en cada paso hay a lo más un
  estado posible, y la notación de configuraciones $(q, w) \vdash (q', w')$ documenta el camino
  completo. Su costo es $O(|w|)$.
- El **AFND** se evalúa por **conjuntos de estados** —la construcción de subconjuntos aplicada al
  vuelo—: en cada paso se toma la unión de los destinos de todos los estados actuales y se le
  aplica la clausura-λ. La cadena se acepta si el conjunto final interseca a $F$.
- Las **transiciones λ** dejan avanzar sin consumir símbolos. Por eso en $a^*b^*c^*$ el conjunto
  inicial ya es $\{q_0, q_1, q_2\}$ y $\lambda$ se acepta sin leer nada.
- AFD y AFND reconocen exactamente la misma clase de lenguajes (los regulares), pero el no
  determinismo hace mucho más natural *describir* ciertos lenguajes: en $(a|b)^*abb$ el AFND
  simplemente «adivina» dónde empieza el sufijo, mientras que el AFD equivalente necesita estados
  extra para recordar cuánto del sufijo lleva reconocido.
- Por eso ambos evaluadores comparten la misma forma de respuesta: cambia la estrategia interna
  (un estado contra un conjunto de estados), no la información que describe al autómata.